In [9]:
from odc.ui import select_on_a_map

geopolygon = select_on_a_map(height='500px', 
                             center=(11.257, 75.787),
                             zoom=10)

Map(center=[11.257, 75.787], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom…

In [10]:
import xarray as xr
import numpy as np
from datacube import Datacube
bbox = list((geopolygon.exterior.coords[0],geopolygon.exterior.coords[3]))
lon = np.round((bbox[0][0],bbox[1][0]),2)
lat = np.round((bbox[0][1],bbox[1][1]),2)
print (lat,lon)



[11.22 11.22] [75.81 75.84]


In [11]:
query = dict(lat=lat,lon=lon,time=("2020-01-01", "2025-12-31"),
    output_crs="EPSG:32643",
    resolution=(-10, 10),
    dask_chunks={"time": 5, "x": 1024, "y": 1024})

In [12]:
dc = Datacube()
ds = dc.load(
    product="sentinel_2_c1_l2a",
    measurements=["red","green","blue","scl","nir"],
    **query
)
print(ds.data_vars)  # or ds.variables


Data variables:
    red      (time, y, x) uint16 441kB dask.array<chunksize=(5, 2, 328), meta=np.ndarray>
    green    (time, y, x) uint16 441kB dask.array<chunksize=(5, 2, 328), meta=np.ndarray>
    blue     (time, y, x) uint16 441kB dask.array<chunksize=(5, 2, 328), meta=np.ndarray>
    scl      (time, y, x) uint8 220kB dask.array<chunksize=(5, 2, 328), meta=np.ndarray>
    nir      (time, y, x) uint16 441kB dask.array<chunksize=(5, 2, 328), meta=np.ndarray>


In [13]:
# Cloud mask (vegetation, bare soil, water, unclassified)
valid_scl = [4, 5, 6, 7]
clean = ds.where(ds.scl.isin(valid_scl))
rgb = clean.drop_vars("scl").median("time")  # lazy with Dask

rgb

<xarray.Dataset> Size: 13kB
Dimensions:      (y: 2, x: 328)
Coordinates:
  * y            (y) float64 16B 1.24e+06 1.24e+06
  * x            (x) float64 3kB 5.884e+05 5.884e+05 ... 5.917e+05 5.917e+05
    spatial_ref  int32 4B 32643
Data variables:
    red          (y, x) float32 3kB dask.array<chunksize=(2, 328), meta=np.ndarray>
    green        (y, x) float32 3kB dask.array<chunksize=(2, 328), meta=np.ndarray>
    blue         (y, x) float32 3kB dask.array<chunksize=(2, 328), meta=np.ndarray>
    nir          (y, x) float32 3kB dask.array<chunksize=(2, 328), meta=np.ndarray>
Attributes:
    crs:           EPSG:32643
    grid_mapping:  spatial_ref

In [7]:
pip install dea-tools


Note: you may need to restart the kernel to use updated packages.


In [14]:
from dea_tools.datahandling import calculate_indices

ImportError: cannot import name 'calculate_indices' from 'dea_tools.datahandling' (/home/batman/miniconda3/envs/cubeenv/lib/python3.13/site-packages/dea_tools/datahandling.py)

In [15]:
pwd